# Chapter 9: Sufficiency and the Rao–Blackwell Theorem

The computational applications of this chapter of *Theory of Statistics*. The code is generated automatically from the book's `code/ch09/` files; see the book for the explanations and proofs.

[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/ibrahimguney/statistical-theory/blob/main/notebooks/ch_09.ipynb)

$$
\newcommand{\R}{\mathbb{R}}\newcommand{\N}{\mathbb{N}}\newcommand{\E}{\mathbb{E}}
\newcommand{\P}{\mathbb{P}}\newcommand{\F}{\mathcal{F}}\newcommand{\I}{\mathbb{I}}
\newcommand{\Var}{\operatorname{Var}}\newcommand{\Cov}{\operatorname{Cov}}
\newcommand{\Corr}{\operatorname{Corr}}\newcommand{\MSE}{\operatorname{MSE}}
\newcommand{\Bias}{\operatorname{Bias}}\newcommand{\Normal}{\mathcal{N}}
\newcommand{\iid}{\overset{\text{i.i.d.}}{\sim}}\newcommand{\pto}{\xrightarrow{P}}
\newcommand{\dto}{\xrightarrow{d}}\newcommand{\X}{\mathbf{X}}\newcommand{\x}{\mathbf{x}}
$$

In [1]:
# Setup: numpy, scipy and matplotlib are preinstalled on Colab. Locally:
#   pip install -r code/requirements.txt
# Figure files (PDF) saved by the code are written to the _figures/ folder; the figures are
# also shown inside the notebook.
import os
os.makedirs("_figures", exist_ok=True)
os.chdir("_figures")
%matplotlib inline

## Bernoulli: $X_1\to\bar X$

*Source file:* `code/ch09/c09_1_bernoulli.py`

In [2]:
import numpy as np

rng = np.random.default_rng(42)
samples = rng.binomial(1, 0.4, size=(200_000, 10))
print(f"Var(W=X_1): {np.var(samples[:,0], ddof=1):.4f} | Var(W*=X_bar): {np.var(np.mean(samples, axis=1), ddof=1):.4f}")
print(f"Theory    : {0.4*0.6:.4f}                | {0.4*0.6/10:.4f}")

Var(W=X_1): 0.2400 | Var(W*=X_bar): 0.0240
Theory    : 0.2400                | 0.0240


## Poisson: Three Estimators of $\P(X=0)$

*Source file:* `code/ch09/c09_2_poisson_three_estimators.py`

In [3]:
import numpy as np

rng = np.random.default_rng(42)
lam, n, R = 2.0, 10, 400_000
X = rng.poisson(lam, size=(R, n))
T = X.sum(axis=1)
target = np.exp(-lam)
estimators = {"W  = I(X1=0)        ": (X[:, 0] == 0).astype(float),
             "W* = (1-1/n)^T (RB) ": (1 - 1/n)**T,
             "exp(-Xbar) (plug-in)": np.exp(-T/n)}
print(f"Target e^-lambda = {target:.5f} | CRLB = {np.exp(-2*lam)*lam/n:.3e}")
for name, t in estimators.items():
    print(f"{name} | mean={t.mean():.5f} | bias={t.mean()-target:+.5f} | Var={t.var():.3e} | MSE={np.mean((t-target)**2):.3e}")
print(f"Theory Var(W*) = {np.exp(-2*lam)*(np.exp(lam/n)-1):.3e}")

Target e^-lambda = 0.13534 | CRLB = 3.663e-03
W  = I(X1=0)         | mean=0.13553 | bias=+0.00020 | Var=1.172e-01 | MSE=1.172e-01
W* = (1-1/n)^T (RB)  | mean=0.13539 | bias=+0.00005 | Var=4.042e-03 | MSE=4.042e-03
exp(-Xbar) (plug-in) | mean=0.14914 | bias=+0.01380 | Var=4.399e-03 | MSE=4.590e-03
Theory Var(W*) = 4.055e-03


## $\text{Uniform}(0,\theta)$: Rao–Blackwell by Simulation

*Source file:* `code/ch09/c09_3_rao_blackwell_simulation.py`

In [4]:
import numpy as np

rng = np.random.default_rng(42)
theta, n, R = 5.0, 8, 300_000
X = rng.uniform(0, theta, size=(R, n))
W = 2 * X[:, 0]
M = X.max(axis=1)
W_star = (n + 1) / n * M
# Check the function E[W | M] empirically: split M into slices
bins = np.quantile(M, np.linspace(0, 1, 6))
idx = np.digitize(M, bins[1:-1])
for k in range(5):
    sel = idx == k
    print(f"M slice {k}: mean(W | M) = {W[sel].mean():.3f}  vs  (n+1)/n * mean(M) = {(n+1)/n*M[sel].mean():.3f}")
print(f"Var(W) = {W.var():.4f} (theory {theta**2/3:.4f}) | Var(W*) = {W_star.var():.4f} (theory {theta**2/(n*(n+2)):.4f})")

M slice 0: mean(W | M) = 4.078  vs  (n+1)/n * mean(M) = 4.089
M slice 1: mean(W | M) = 4.822  vs  (n+1)/n * mean(M) = 4.827
M slice 2: mean(W | M) = 5.143  vs  (n+1)/n * mean(M) = 5.153
M slice 3: mean(W | M) = 5.381  vs  (n+1)/n * mean(M) = 5.377
M slice 4: mean(W | M) = 5.553  vs  (n+1)/n * mean(M) = 5.550
Var(W) = 8.3277 (theory 8.3333) | Var(W*) = 0.3123 (theory 0.3125)
